In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.preprocessing import MinMaxScaler
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

2025-12-17 03:18:29.970894: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765941510.130196      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765941510.178513      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765941510.550430      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765941510.550467      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765941510.550470      55 computation_placer.cc:177] computation placer alr

In [3]:
# ==============================================================================
# 1. KONFIGURASI
# ==============================================================================
PATH_FILE = '/kaggle/input/open_meteo_climate/kebumen_75tahun_lengkap.csv'

In [4]:
# ==============================================================================
# 2. LOAD DATA & BERSIHKAN WAKTU
# ==============================================================================
def load_data(filepath):
    print(f"📂 Membaca file: {filepath}")
    if not os.path.exists(filepath):
        print("❌ File tidak ditemukan.")
        return None
    
    # Baca CSV
    df = pd.read_csv(filepath, index_col='date', parse_dates=True)
    
    # Perbaikan Zona Waktu (Agar tidak geser)
    # 1. Baca sebagai UTC
    df.index = pd.to_datetime(df.index, utc=True)
    # 2. Geser ke WIB
    df.index = df.index.tz_convert('Asia/Jakarta')
    # 3. Hapus label zona waktu (biar jadi jam dinding polos)
    df.index = df.index.tz_localize(None)
    
    df = df.sort_index()
    print(f"✅ Data Siap! Rentang: {df.index.min()} s.d {df.index.max()}")
    return df

df = load_data(PATH_FILE)

df.tail(10)

📂 Membaca file: /kaggle/input/open_meteo_climate/kebumen_75tahun_lengkap.csv
✅ Data Siap! Rentang: 1950-01-01 01:00:00 s.d 2025-12-16 23:00:00


,temperature,humidity,dewpoint,rain_mm,wind_speed,wind_direction,pressure,weather_code
date,,,,,,,,
2025-12-16 14:00:00,29.788,68.168320,23.288000,0.1,20.531050,242.31174,1005.04460,51.0
2025-12-16 15:00:00,29.788,67.145905,23.038000,0.0,20.366266,243.20856,1004.44586,2.0
2025-12-16 16:00:00,29.188,68.881805,22.888000,0.0,19.111390,240.05441,1004.74120,1.0
2025-12-16 17:00:00,28.588,71.534355,22.938000,0.0,15.250155,239.50024,1004.83655,3.0
2025-12-16 18:00:00,28.088,71.011910,22.338000,0.1,14.892213,243.43501,1005.43176,51.0
2025-12-16 19:00:00,28.538,70.449470,22.638000,0.0,19.121557,236.90830,1006.23320,3.0
2025-12-16 20:00:00,27.888,73.835150,22.788000,0.0,19.319628,243.43501,1006.82720,3.0
2025-12-16 21:00:00,25.838,85.078090,23.138000,0.7,14.535487,262.17102,1007.61060,53.0
2025-12-16 22:00:00,24.438,94.451645,23.487999,2.0,8.557102,284.62090,1008.09910,61.0


In [5]:
# Ambil data 10 tahun terakhir agar relevan dan training tidak terlalu lama
# (Silakan ubah tahunnya jika mau lebih banyak data)
df_ai = df.loc['1950':].copy()

In [6]:
df_ai.tail(10)

,temperature,humidity,dewpoint,rain_mm,wind_speed,wind_direction,pressure,weather_code
date,,,,,,,,
2025-12-16 14:00:00,29.788,68.168320,23.288000,0.1,20.531050,242.31174,1005.04460,51.0
2025-12-16 15:00:00,29.788,67.145905,23.038000,0.0,20.366266,243.20856,1004.44586,2.0
2025-12-16 16:00:00,29.188,68.881805,22.888000,0.0,19.111390,240.05441,1004.74120,1.0
2025-12-16 17:00:00,28.588,71.534355,22.938000,0.0,15.250155,239.50024,1004.83655,3.0
2025-12-16 18:00:00,28.088,71.011910,22.338000,0.1,14.892213,243.43501,1005.43176,51.0
2025-12-16 19:00:00,28.538,70.449470,22.638000,0.0,19.121557,236.90830,1006.23320,3.0
2025-12-16 20:00:00,27.888,73.835150,22.788000,0.0,19.319628,243.43501,1006.82720,3.0
2025-12-16 21:00:00,25.838,85.078090,23.138000,0.7,14.535487,262.17102,1007.61060,53.0
2025-12-16 22:00:00,24.438,94.451645,23.487999,2.0,8.557102,284.62090,1008.09910,61.0


In [10]:
df_ai.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 622032 entries, 0 to 622031
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   temperature     622032 non-null  float64
 1   humidity        622032 non-null  float64
 2   dewpoint        622032 non-null  float64
 3   rain_mm         622032 non-null  float64
 4   wind_speed      622032 non-null  float64
 5   wind_direction  622032 non-null  float64
 6   pressure        622032 non-null  float64
 7   weather_code    622032 non-null  float64
 8   target_class    622032 non-null  int64  
dtypes: float64(8), int64(1)
memory usage: 42.7 MB


In [19]:
# ==============================================================================
# 3. FEATURE ENGINEERING (KELAS & MULTI-TIME LAG)
# ==============================================================================

# --- A. PEMBAGIAN KELAS (4 LEVEL) ---
def categorize_rain(mm):
    if mm < 0.5:
        return 0  # KELAS 0: Aman (Cerah/Berawan)
    elif mm < 10.0:
        return 1  # KELAS 1: Hujan Biasa (Ringan-Sedang)
    else:
        return 2  # KELAS 2: BAHAYA (Lebat-Badai)

# Terapkan kategori ke kolom target baru
df_ai['target_class'] = df_ai['rain_mm'].apply(categorize_rain)

# --- B. TIME LAG (FITUR MASA LALU UNTUK BANYAK KOLOM) ---
# 1. Tentukan kolom mana saja yang mau dibuatkan "memori"-nya
# Ganti nama kolom di bawah ini sesuai dengan nama kolom di data kamu!
#fitur_untuk_lag = ['rain_mm', 'temperature', 'humidity', 'pressure','wind_speed'] 

# 2. Tentukan mau mundur berapa jam (misal: 1, 2, dan 3 jam lalu)
#lags = [1, 2, 3, 4, 6, 12, 24] 

# 3. Loop otomatis untuk membuat kolom baru
#for col in fitur_untuk_lag:
    # Cek dulu apakah kolomnya ada di data biar tidak error
#    if col in df_ai.columns:
 #       for lag in lags:
            # Nama kolom baru: misal 'T2m_lag_1', 'RH_lag_2'
#            col_name = f"{col}_lag_{lag}"
 #           df_ai[col_name] = df_ai[col].shift(lag)
 #   else:
#        print(f"⚠️ Peringatan: Kolom '{col}' tidak ditemukan di data, dilewati.")

# --- C. PEMBERSIHAN (CLEANING) ---
# Hapus baris NaN akibat pergeseran data (shift)
df_ai = df_ai.dropna().reset_index(drop=True)

# --- CEK HASIL ---
print(f"\n✅ Total Data setelah lag & cleaning: {len(df_ai)} baris")
print(f"📊 Jumlah Fitur (Kolom) Sekarang: {len(df_ai.columns)}")

# Tampilkan contoh kolom T2m (Suhu) dan lag-nya
if 'temperature ' in df_ai.columns:
    cols_show = ['temperature ', 'T2m_lag_1', 'T2m_lag_2'] 
    print("\nContoh Lag pada Suhu (T2m):")
    print(df_ai[cols_show].head(5))


✅ Total Data setelah lag & cleaning: 622032 baris
📊 Jumlah Fitur (Kolom) Sekarang: 9


In [23]:
# ==============================================================================
# 4. PRE-PROCESSING (SCALING & WINDOWING)
# ==============================================================================
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# --- A. MENENTUKAN FITUR (X) & TARGET (y) ---

# 1. Fitur Utama (Sesuai request kamu)
fitur_utama = ['rain_mm', 'temperature', 'humidity', 'pressure','wind_speed','wind_direction']

# 2. Ambil otomatis semua kolom yang mengandung nama fitur utama + kolom lag-nya
# Ini trik biar kita tidak usah ngetik manual 'rain_lag_1', 'rain_lag_2', dst.
features = [col for col in df_ai.columns if any(x in col for x in fitur_utama)]

# 3. Target Output (Kita pakai target_class yang sudah dibuat di step 3)
target = 'target_class' 

print(f"✅ Jumlah Fitur Input (X): {len(features)} kolom")
print(f"   (Termasuk fitur asli dan semua lag yang telah dibuat)")
print(f"✅ Target Label (y): {target}")

# --- B. SCALING (MENYAMAKAN SKALA 0-1) ---
scaler = MinMaxScaler()
# Kita scale hanya kolom fitur (X), target (y) tidak perlu di-scale karena klasifikasi
X_scaled = scaler.fit_transform(df_ai[features])
y_raw = df_ai[target].values

# --- C. WINDOWING (MEMOTONG DATA BERDASARKAN WAKTU) ---
def create_windows(X, y, time_steps=1):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y[i + time_steps])
    return np.array(Xs), np.array(ys)

# PENTING:
# Karena kita SUDAH punya fitur Lag (masa lalu) di kolom, 
# kita bisa set TIME_STEPS = 1 (atau angka kecil).
# Tapi jika kamu mau model melihat "urutan" data lag tersebut, kita bisa pakai angka lebih besar.
# Mari kita coba pakai 1 dulu karena kolomnya sudah mengandung informasi masa lalu (24 jam).
TIME_STEPS = 24

print(f"\n✂️ Memotong data (Windowing) dengan Time Steps: {TIME_STEPS}...")
X_window, y_window = create_windows(X_scaled, y_raw, TIME_STEPS)

# --- D. SPLITTING DATA (LATIH & UJI) ---
# Split Training (80%) dan Testing (20%) - Tanpa mengacak urutan waktu
split_idx = int(len(X_window) * 0.80)

X_train, y_train = X_window[:split_idx], y_window[:split_idx]
X_test, y_test = X_window[split_idx:], y_window[split_idx:]

print(f"\n📊 Bentuk Data Siap Training:")
print(f"Data Latih (X_train): {X_train.shape}")
print(f"Label Latih (y_train): {y_train.shape}")
print(f"Data Uji  (X_test) : {X_test.shape}")

✅ Jumlah Fitur Input (X): 6 kolom
   (Termasuk fitur asli dan semua lag yang telah dibuat)
✅ Target Label (y): target_class

✂️ Memotong data (Windowing) dengan Time Steps: 24...

📊 Bentuk Data Siap Training:
Data Latih (X_train): (497606, 24, 6)
Label Latih (y_train): (497606,)
Data Uji  (X_test) : (124402, 24, 6)


In [24]:
# Anggap y_train adalah label datamu (0: Cerah, 1: Hujan Ringan, 2: Badai)
# y_train = [0, 0, 0, 1, 0, 2, 0, 0, 1, ...] 

# 1. Hitung bobot otomatis agar seimbang
class_weights_vals = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# 2. Ubah jadi Dictionary (Format yang diminta Keras)
class_weights_dict = dict(enumerate(class_weights_vals))

print("Bobot per Kelas:", class_weights_dict)
# Output mungkin seperti: {0: 0.5, 1: 2.3, 2: 15.8}
# Artinya: Kelas 2 (Badai) dianggap 15x lebih penting/berbobot.

Bobot per Kelas: {0: np.float64(0.3886614694756804), 1: np.float64(2.3537486400832504), 2: np.float64(451.95821980018167)}


In [29]:
# ==============================================================================
# 6. MEMBANGUN & MELATIH MODEL (UPGRADED ARCHITECTURE - 3 CLASS)
# ==============================================================================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# --- A. DEFINISI ARSITEKTUR MODEL (IMPROVISED) ---
model = Sequential()

# 1. Input Layer
model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))

# 2. LSTM Layer 1 (Awal)
# return_sequences=True: TERUSKAN data urutan waktu ke layer bawahnya
model.add(LSTM(80, return_sequences=True))
model.add(Dropout(0.3))

# 3. LSTM Layer 2 (Tengah)
# ⚠️ PERBAIKAN: Ubah jadi True agar bisa masuk ke LSTM ke-3
model.add(LSTM(40, return_sequences=True)) 
model.add(Dropout(0.3))

# 4. LSTM Layer 3 (Akhir)
# return_sequences=False: STOP urutan waktu, gepengkan data masuk ke Dense
model.add(LSTM(40, return_sequences=False)) 
model.add(Dropout(0.3))

# 5. Dense Layer
model.add(Dense(32, activation='relu'))
model.add(BatchNormalization())

# 6. Output Layer (3 KELAS)
model.add(Dense(3, activation='softmax'))

# --- B. COMPILE MODEL ---
# Kita definisikan ulang metrik agar lengkap
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001, clipvalue=1.0), # Kita set learning rate eksplisit
    loss='sparse_categorical_crossentropy', # Wajib untuk 3 kelas (0,1,2)
    metrics=['accuracy']
)

model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_15 (LSTM)                  │ (None, 24, 80)         │        27,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 24, 80)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_16 (LSTM)                  │ (None, 24, 40)         │        19,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 24, 40)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_17 (LSTM)                  │ (None, 40)             │        12,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 40)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         1,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,699 (241.01 KB)

 Trainable params: 61,635 (240.76 KB)

 Non-trainable params: 64 (256.00 B)

In [ ]:
# ==============================================================================
# 6. TRAINING (MELATIH OTAK AI)
# ==============================================================================
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# B. Reduce LR: Kalau stuck, turunkan learning rate biar lebih teliti
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=0.00001, verbose=1)

# C. Checkpoint: Simpan model terbaik selama proses
checkpoint = ModelCheckpoint('model_terbaik.keras', monitor='val_accuracy', save_best_only=True, verbose=1)

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=128,
    validation_split=0.1,
    class_weight=class_weights_dict,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)
# Plot History Belajar
plt.figure(figsize=(10, 4))
plt.plot(history.history['accuracy'], label='Akurasi Latih')
plt.plot(history.history['val_accuracy'], label='Akurasi Validasi')
plt.title('Grafik Pembelajaran Model')
plt.legend()
plt.show()

Epoch 1/100
3498/3499 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4471 - loss: 1.0993
Epoch 1: val_accuracy improved from -inf to 0.57091, saving model to model_terbaik.keras
3499/3499 ━━━━━━━━━━━━━━━━━━━━ 38s 10ms/step - accuracy: 0.4471 - loss: 1.0992 - val_accuracy: 0.5709 - val_loss: 0.8971 - learning_rate: 1.0000e-04
Epoch 2/100
3497/3499 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4985 - loss: 0.9835
Epoch 2: val_accuracy improved from 0.57091 to 0.61978, saving model to model_terbaik.keras
3499/3499 ━━━━━━━━━━━━━━━━━━━━ 34s 10ms/step - accuracy: 0.4985 - loss: 0.9835 - val_accuracy: 0.6198 - val_loss: 0.8264 - learning_rate: 1.0000e-04
Epoch 3/100
3495/3499 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5253 - loss: 0.9624
Epoch 3: val_accuracy did not improve from 0.61978
3499/3499 ━━━━━━━━━━━━━━━━━━━━ 34s 10ms/step - accuracy: 0.5253 - loss: 0.9624 - val_accuracy: 0.5799 - val_loss: 0.8332 - learning_rate: 1.0000e-04
Epoch 4/100
3494/3499 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step

In [ ]:
# ==============================================================================
# 7. EVALUASI HASIL (VERSI 3 KELAS)
# ==============================================================================
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

print("\n📊 EVALUASI MODEL:")

# 1. Prediksi Data Test
y_pred_prob = model.predict(X_test)
y_pred_class = np.argmax(y_pred_prob, axis=1)

# 2. DEFINISIKAN NAMA KELAS (INI YANG KURANG TADI)
# Sesuai keputusan terakhir: 3 Kelas
CLASSES = {
    0: 'Berawan', 
    1: 'Hujan Ringan', 
    2: 'Hujan Sedang-Lebat'
}
target_names = list(CLASSES.values()) # Mengambil ['Berawan', 'Hujan Ringan', ...]

# 3. Laporan Klasifikasi
print(classification_report(y_test, y_pred_class, target_names=target_names))

# 4. Confusion Matrix
cm = confusion_matrix(y_test, y_pred_class)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, 
            fmt='d', 
            cmap='Blues', 
            xticklabels=target_names)
plt.xlabel('Prediksi AI')
plt.ylabel('Kenyataan (Aktual)')
plt.title('Confusion Matrix (Evaluasi Ketepatan)')

# Simpan Gambar
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight') 
print("✅ Gambar berhasil disimpan sebagai 'confusion_matrix.png'")
plt.show()

In [ ]:
# ==============================================================================
# 8. DEMO PREDIKSI (KARTU CUACA - VERSI 3 KELAS)
# ==============================================================================
import numpy as np

# 1. Definisikan Label sesuai pembagian baru
CLASSES = {
    0: 'Berawan / Kering', 
    1: 'Hujan Ringan', 
    2: 'Hujan Sedang-Lebat'
}

def tampilkan_kartu_prediksi(input_data):
    """Menampilkan prediksi dengan gaya persentase."""
    # Pastikan bentuk input sesuai dengan yang diminta model (1, time_steps, features)
    # Kita ambil bentuk dimensi dari X_test langsung biar aman
    input_reshaped = input_data.reshape(1, X_test.shape[1], X_test.shape[2])
    
    # Prediksi
    probs = model.predict(input_reshaped, verbose=0)[0]
    pred_idx = np.argmax(probs)
    confidence = probs[pred_idx] * 100
    
    # Tampilan Visual
    print("\n" + "="*50)
    print(f"🔮 PREDIKSI CUACA (1 Jam Kedepan)")
    print("="*50)
    
    print(f"Hasil Utama    : {CLASSES[pred_idx]}")
    print(f"Tingkat Yakin  : {confidence:.1f}%")
    
    # Tampilkan Data Input Penting (Opsional, buat ngecek)
    # Kita ambil data terakhir dari sequence (time step terakhir)
    # Kolom 0 biasanya rain_mm, tapi karena sudah di-scale jadi susah dibaca manusia.
    # Jadi kita skip menampilkan angka mentah, fokus ke probabilitas saja.
    
    print("-" * 50)
    print("Probabilitas per Kelas:")
    for idx, label in CLASSES.items():
        score = probs[idx] * 100
        # Bikin grafik batang sederhana
        bar_length = int(score / 4) 
        bar = "█" * bar_length
        marker = "👈 Pilihan AI" if idx == pred_idx else ""
        
        # Format print yang rapi
        print(f"{label:<25} : {score:>5.1f}% | {bar} {marker}")
    print("="*50)

# --- CARI CONTOH KASUS HUJAN LEBAT ---
print("\n🧪 DEMO HASIL PADA DATA HUJAN LEBAT (KELAS 2):")

# Kita cari data di X_test yang aslinya adalah Kelas 2 (Sedang-Lebat)
# y_test berisi label asli. Kita cari indeks di mana nilainya == 2.
badai_indices = np.where(y_test == 2)[0]

if len(badai_indices) > 0:
    # Ambil 1 contoh saja (misal contoh pertama yang ditemukan)
    sample_idx = badai_indices[0]
    
    print(f"Menguji data test index ke-{sample_idx}...")
    tampilkan_kartu_prediksi(X_test[sample_idx])
    
    print(f"\n(Kunci Jawaban Asli: {CLASSES[y_test[sample_idx]]})")
else:
    print("Kebetulan tidak ada data Hujan Sedang-Lebat di potongan data test ini.")
    # Kalau tidak ada yang lebat, coba tampilkan hujan ringan (Kelas 1)
    ringan_indices = np.where(y_test == 1)[0]
    if len(ringan_indices) > 0:
        print("\nMenampilkan contoh Hujan Ringan saja:")
        tampilkan_kartu_prediksi(X_test[ringan_indices[0]])